<a href="https://colab.research.google.com/github/zombimann/Mathematical-video-animations-and-visualization/blob/main/Complex_Hamiltonian_surface_with_projections_visualization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Hamiltonian Surface Dynamics Animation

**Author:** Mugambi Ndwiga  
**Organization:** Zoom Bee Apps

## Overview
This project visualizes classical Hamiltonian dynamics on a complex energy landscape. It simulates a particle's motion using a 4th-order Runge-Kutta integrator to ensure energy conservation and renders the results as a high-quality 3D animation.

## Features
- **Complex Hamiltonian:** A custom energy surface composed of Gaussian peaks and valleys.
- **Phase Space Projections:** Real-time shadow projections onto the (q, p) bottom plane and the (H, p) back plane.
- **Energy Conservation:** High-precision integration ensuring the particle stays locked to its iso-energy contour.
- **Cinematic Visualization:** Dynamic camera rotation and a sleek 'energetic' color scheme.

## Requirements
- `numpy`
- `matplotlib`
- `scipy` (for potential grid-based calculations)
- `IPython` (for notebook embedding)

## Usage
Simply run the code cell below in a Google Colab or Jupyter environment. The animation will render as a compressed HTML5 video (MP4) for optimal performance.

In [7]:
"""
Hamiltonian Surface Animation
Author: Mugambi Ndwiga, Zoom Bee Apps

This script simulates and animates a particle's trajectory on a complex
Hamiltonian energy surface H(q, p). It uses a 4th-order Runge-Kutta
integrator to conserve energy and visualizes the dynamics with 3D
surfaces, contour projections, and real-time tracking dots.
"""

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.colors import LinearSegmentedColormap
from IPython.display import HTML
import matplotlib

# Set a high limit for the embedded animation
matplotlib.rcParams['animation.embed_limit'] = 150.0

# ── 1. Configuration ─────────────────────────────────────────────────────────
CFG = dict(
    FPS         = 25,
    DURATION    = 15,
    SPEED       = 2.5,
    GRID        = 40,
    RANGE       = 5.0,
    ORBIT_STEPS = 5000,
    DT          = 0.02,
    TRAIL_LEN   = 80,
    N_CONTOURS  = 15,
    PROJ_Z      = -2.0,     # Bottom plane Z
    PROJ_Q      = -5.0,     # Far side 'Back' plane
    CAM_ELEV    = 30,
    CAM_AZIM0   = -45,
    CAM_ROT     = 360,
)
CFG['N_FRAMES'] = CFG['FPS'] * CFG['DURATION']

# ── 2. Complex Hamiltonian ──────────────────────────────────────────────────
def H(q, p):
    ke = 0.5 * p**2
    pe = 0.1 * (q**2)
    gaussians = [
        (1.5, 1.2, 0.5), (1.2, -1.0, 0.7), (1.0, -2.5, 0.4), (-0.8, 2.5, 0.6)
    ]
    for amp, mu, sig in gaussians:
        pe += amp * np.exp(-((q - mu)**2) / (2 * sig**2))
    return ke + pe

def dHdq(q, p, dq=1e-4):
    return (H(q + dq, p) - H(q - dq, p)) / (2 * dq)

# ── 3. Physics Integration ──────────────────────────────────────────────────
def rk4_step(q, p, dt):
    k1q, k1p = p, -dHdq(q, p)
    k2q, k2p = p + 0.5*dt*k1p, -dHdq(q + 0.5*dt*k1q, p + 0.5*dt*k1p)
    k3q, k3p = p + 0.5*dt*k2p, -dHdq(q + 0.5*dt*k2q, p + 0.5*dt*k2p)
    k4q, k4p = p + dt*k3p, -dHdq(q + dt*k3q, p + dt*k3p)
    return q + dt/6*(k1q + 2*k2q + 2*k3q + k4q), p + dt/6*(k1p + 2*k2p + 2*k3p + k4p)

q_track, p_track = np.zeros(CFG['ORBIT_STEPS']), np.zeros(CFG['ORBIT_STEPS'])
cur_q, cur_p = 2.0, 0.5
for i in range(CFG['ORBIT_STEPS']):
    q_track[i], p_track[i] = cur_q, cur_p
    cur_q, cur_p = rk4_step(cur_q, cur_p, CFG['DT'])
h_track = H(q_track, p_track)

# ── 4. Scene Setup ──────────────────────────────────────────────────────────
fig = plt.figure(figsize=(12, 8), facecolor='#050505')
ax = fig.add_subplot(111, projection='3d', facecolor='#050505')

R = CFG['RANGE']
lin = np.linspace(-R, R, CFG['GRID'])
QQ, PP = np.meshgrid(lin, lin)
ZZ = H(QQ, PP)

cmap = LinearSegmentedColormap.from_list('energetic', ['#000033', '#00ccff', '#ffcc00', '#ff3300'])
surf = ax.plot_surface(QQ, PP, ZZ, cmap=cmap, alpha=0.6, linewidth=0, antialiased=True, zorder=1)

pz, pq = CFG['PROJ_Z'], CFG['PROJ_Q']
ax.contour(QQ, PP, ZZ, levels=CFG['N_CONTOURS'], zdir='z', offset=pz, cmap='cool', alpha=0.4)
ax.contour(QQ, PP, ZZ, levels=CFG['N_CONTOURS'], zdir='x', offset=pq, cmap='autumn', alpha=0.4)

trail, = ax.plot([], [], [], color='white', lw=1.5, alpha=0.8, zorder=10)
particle, = ax.plot([], [], [], 'o', color='white', ms=8, mec='cyan', mew=2, zorder=11)
# Dynamic projection dots
proj_z_dot, = ax.plot([], [], [], 'o', color='cyan', ms=5, alpha=0.8, zorder=5)
proj_q_dot, = ax.plot([], [], [], 'o', color='orange', ms=5, alpha=0.8, zorder=5)

v_line, = ax.plot([], [], [], '--', color='cyan', lw=0.5, alpha=0.6)
h_line, = ax.plot([], [], [], '--', color='orange', lw=0.5, alpha=0.6)

ax.set_xlim(pq, R); ax.set_ylim(-R, R); ax.set_zlim(pz, ZZ.max() + 1)
ax.axis('off')

# ── 5. Animation ────────────────────────────────────────────────────────────
def update(frame):
    idx = int(frame * CFG['SPEED']) % CFG['ORBIT_STEPS']
    q, p, h = q_track[idx], p_track[idx], h_track[idx]
    start = max(0, idx - CFG['TRAIL_LEN'])
    trail.set_data(q_track[start:idx], p_track[start:idx])
    trail.set_3d_properties(h_track[start:idx])
    particle.set_data([q], [p])
    particle.set_3d_properties([h])

    # Update dynamic projection dots
    proj_z_dot.set_data([q], [p])
    proj_z_dot.set_3d_properties([pz])
    proj_q_dot.set_data([pq], [p])
    proj_q_dot.set_3d_properties([h])

    v_line.set_data([q, q], [p, p])
    v_line.set_3d_properties([h, pz])
    h_line.set_data([q, pq], [p, p])
    h_line.set_3d_properties([h, h])
    ax.view_init(elev=CFG['CAM_ELEV'], azim=CFG['CAM_AZIM0'] + (frame/CFG['N_FRAMES']) * CFG['CAM_ROT'])
    return particle, trail, v_line, h_line, proj_z_dot, proj_q_dot

anim = FuncAnimation(fig, update, frames=CFG['N_FRAMES'], interval=1000/CFG['FPS'], blit=False)
plt.close(fig)
HTML(anim.to_html5_video())